# TrendPulse Configuration

Set dataset paths, countries, and analysis thresholds before loading data. Dataset paths are resolved from the repository root, so this notebook can also be run from a subfolder.

In [1]:
from datetime import datetime
from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "dataset").is_dir()
)
FILE_PATHS: list[Path] = [
    PROJECT_ROOT / "dataset/GBvideos.csv",
    PROJECT_ROOT / "dataset/INvideos.csv",
    PROJECT_ROOT / "dataset/USvideos.csv",
]
CATEGORIES_PATH: Path = PROJECT_ROOT / "dataset/categories.csv"
COUNTRIES: list[str] = ["GB", "IN", "US"]

MIN_VIEWS: int = 1_000_000
MIN_LIKES: int = 10_000
ENGAGEMENT_RATE_THRESHOLD: float = 0.05

RUN_STARTED_AT: datetime = datetime.now().astimezone()

In [2]:
print("=" * 60)
print(f"TrendPulse data load | started: {RUN_STARTED_AT:%Y-%m-%d %H:%M:%S %Z}")
print(f"Countries to load ({len(COUNTRIES)}): {', '.join(COUNTRIES)}")
print("=" * 60)

TrendPulse data load | started: 2026-10-26 18:23:42 IST
Countries to load (3): GB, IN, US


## Loading country data

Load each country CSV into records, checking that the expected video fields are present. The optional `max_rows` argument can limit a load for quick testing.

In [3]:
REQUIRED_VIDEO_COLUMNS: set[str] = {
    "video_id",
    "trending_date",
    "title",
    "channel_title",
    "category_id",
    "publish_time",
    "tags",
    "views",
    "likes",
    "dislikes",
    "comment_count",
    "thumbnail_link",
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",
    "description",
}


def load_country(path, country, max_rows=None):
    df = pd.read_csv(path, nrows=max_rows)
    missing_columns = REQUIRED_VIDEO_COLUMNS.difference(df.columns)
    if missing_columns:
        raise ValueError(
            f"{path} is missing required columns: {', '.join(sorted(missing_columns))}"
        )

    df["country"] = country
    records = df.to_dict("records")
    print(f"Loaded {len(records):,} records for {country}.")
    return records



In [4]:
categories_df = pd.read_csv(CATEGORIES_PATH)
required_category_columns = {"category_id", "category_name"}
missing_category_columns = required_category_columns.difference(categories_df.columns)
if missing_category_columns:
    raise ValueError(
        f"{CATEGORIES_PATH} is missing required columns: "
        f"{', '.join(sorted(missing_category_columns))}"
    )

category_names: dict[int, str] = categories_df.set_index("category_id")["category_name"].to_dict()
category_names

{1: 'Film & Animation',
 2: 'Autos & Vehicles',
 10: 'Music',
 15: 'Pets & Animals',
 17: 'Sports',
 19: 'Travel & Events',
 20: 'Gaming',
 22: 'People & Blogs',
 23: 'Comedy',
 24: 'Entertainment',
 25: 'News & Politics',
 26: 'Howto & Style',
 27: 'Education',
 28: 'Science & Technology',
 29: 'Nonprofits & Activism',
 30: 'Movies',
 43: 'Shows'}

In [6]:
videos_by_country = {
    country: load_country(path, country) for path, country in zip(FILE_PATHS, COUNTRIES)
}
record_counts = {
    country: len(records) for country, records in videos_by_country.items()
}
record_counts

Loaded 15,075 records for GB.
Loaded 15,075 records for IN.
Loaded 16,080 records for US.


{'GB': 15075, 'IN': 15075, 'US': 16080}

## Notebook discipline

This notebook uses clear headings, is intended to be run with **Restart and Run All**, and avoids hidden state by defining configuration before use.